In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !pip install unsloth

In [ ]:
# import unsloth
# from unsloth import FastLanguageModel
# import torch
# import os

# # Disable Unsloth's attention optimizations
# os.environ["UNSLOTH_RETURN_PT_ONLY"] = "1"

# MODEL_PATH = r"D:\khmer_spell_check\spellcheck_textsummarize\Qwen2.5-1.5B-Khmer-spell-check-20260310T014415Z-1-002"

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MODEL_PATH,
#     max_seq_length=2048,
#     dtype=torch.float16,
#     load_in_4bit=False,
#     attn_implementation=None, # ⭐ IMPORTANT
#     # attn_implementation="flash_attention_2",
# )

# # FastLanguageModel.for_inference(model)

# model.eval()

# print("Model loaded successfully with Unsloth!")

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import os

os.environ["TRANSFORMERS_VERBOSITY"] = "error"  # Suppress transformers warnings
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_PATH = r"D:\khmer_spell_check\spellcheck_textsummarize\Qwen2.5-1.5B-Khmer-spell-check-20260310T014415Z-1-002"

# Load with standard transformers
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("Model loaded successfully with transformers!")

W0327 14:47:42.235000 13032 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Model loaded successfully with transformers!


In [ ]:
# ==============================
# Inference function
# ==============================
SYSTEM_MSG = "You are a Khmer spell corrector. Only output the corrected sentence. No explanations."


def correct_khmer(sentence: str):
    sentence = sentence.strip()
    if sentence == "":
        return ""

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": f"Incorrect: {sentence}\nCorrect:\n"},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.05,
        )

    gen_ids = output[0][input_len:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # -------------------------
    # post-processing
    # -------------------------

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return sentence

    final = lines[0]

    for stop in [" គឺ", " ដែល", " ដោយ", " និង", "៖", ":"]:
        if stop in final:
            final = final.split(stop, 1)[0].strip()

        # -------------------------
        # safety fallback
        # -------------------------
        in_len = len(sentence)
        out_len = len(final)

        # if model rewrites / expands too much → fallback
        if out_len == 0:
            return sentence

        if out_len > int(in_len * 1.3) or out_len < int(in_len * 0.5):
            return sentence

    return final

In [ ]:
# ==============================
# Test
# ==============================
test_sentence = "កូនសិស្សនៅក្នុងថ្នាកសិក្សា"
corrected = correct_khmer(test_sentence)
print("Original:", test_sentence)
print("Corrected:", corrected)

Original: កូនសិស្សនៅក្នុងថ្នាកសិក្សា
Corrected: កូនសិស្សនៅក្នុងថ្នាក់សិក្សា


In [ ]:
# tests = [
#  "គាតជាបងប្អូនខ្ញុំ",
#  "សាលារៀណជាកន្លែងរៀណសូត្រ",
#  "លោគគ្រូខិតខំបង្រៀននិស្សិតគ្របគ្នា",
#  "និស្សិទត្រូវខិតខំរៀនសូត្រ",
#  "ខ្ញុំស្រឡាញ់ប្រទេសកម្ពូជា",
#  "ខ្ញុំចូលចិតស្ដាប់តន្ត្រី",
#  "គាត់កំពុងធ្វេការងារ",
#  "គាត់ចូលចិតលេងបាល",
#  "សិស្សរៀនមេរៀនថ្ម",
#  "កូនសិស្សនៅក្នុងថ្នាកសិក្សា",
# ]
# test=[
#  "សិស្សរៀនមេរៀនថ្ម",
# ]
# # tests1 = [
# #     "ខ្ញុំកំពុងរៀនភាសាខ្មែ",
# #     "ខ្ញុំចូលចិតស្ដាប់តន្ត្រី",
# #     "ខ្ញុំទៅលេងមិត្តភក្ត",
# #     "សាលារៀននេះធំណាស",
# #     "ខ្ញុំចូលចិតញុំាអាហារ",
# #     "គាត់ចូលចិតអានកាសែត",
# #     "សិស្សកំពុងស្ដាបគ្រូ",
# #     "គាត់ចូលចិតលេងបាល់",
# #     "សាលារៀនមានសិស្សច្រើនណ",
# #     "គាត់កំពុងធ្វេការងារ",
# # ]

# print("\n==== Batch test ====\n")

# for i, s in enumerate(tests, 1):
#  out = correct_khmer(s)
#  print(f"{i}. Input : {s}")
#  print(f" Output: {out}")
#  print()

In [ ]:
# # Evaluation test cases
# evaluation_pairs = [
#  # First batch
#  ("គាតជាបងប្អូនខ្ញុំ","គាត់ជាបងប្អូនខ្ញុំ"),
#  ("សាលារៀណជាកន្លែងរៀណសូត្រ","សាលារៀនជាកន្លែងរៀនសូត្រ"),
#  ("លោគគ្រូខិតខំបង្រៀននិស្សិត","លោកគ្រូខិតខំបង្រៀននិស្សិត"),
#  ("និស្សិទត្រូវខិតខំរៀនសូត្រ","និស្សិតត្រូវខិតខំរៀនសូត្រ"),
#  ("ខ្ញុំស្រឡាញ់ប្រទេសកម្ពូជា","ខ្ញុំស្រឡាញ់ប្រទេសកម្ពុជា"),
#  ("ពួកគេទៅសាលារៀនរាល់ថ្ងែ","ពួកគេទៅសាលារៀនរាល់ថ្ងៃ"),
#  ("ខ្ញុំចូលចិតអានសៀវភៅ","ខ្ញុំចូលចិត្តអានសៀវភៅ"),
#  ("សិស្សកំពុងរៀនក្នងថ្នាក់","សិស្សកំពុងរៀនក្នុងថ្នាក់"),
#  ("ម្តាយខ្ញុំកំពុងធ្វេម្ហូប","ម្តាយខ្ញុំកំពុងធ្វើម្ហូប"),
#  ("កុមារតូចៗចូលចិតលេង","កុមារតូចៗចូលចិត្តលេង"),

#  # Second batch
#  ("ខ្ញុំកំពុងរៀនភាសាខ្មែ","ខ្ញុំកំពុងរៀនភាសាខ្មែរ"),
#  ("ខ្ញុំចូលចិតស្ដាប់តន្ត្រី","ខ្ញុំចូលចិត្តស្តាប់តន្ត្រី"),
#  ("ខ្ញុំទៅលេងមិត្តភក្ត","ខ្ញុំទៅលេងមិត្តភក្តិ"),
#  ("សាលារៀននេះធំណាស","សាលារៀននេះធំណាស់"),
#  ("ខ្ញុំចូលចិតញុំាអាហារ","ខ្ញុំចូលចិត្តញុំាអាហារ"),
#  ("គាត់ចូលចិតអានកាសែត","គាត់ចូលចិត្តអានកាសែត"),
#  ("សិស្សកំពុងស្ដាបគ្រូ","សិស្សកំពុងស្តាប់គ្រូ"),
#  ("គាត់ចូលចិតលេងបាល់","គាត់ចូលចិត្តលេងបាល់"),
#  ("សាលារៀនមានសិស្សច្រើនណ","សាលារៀនមានសិស្សច្រើនណាស់"),
#  ("គាត់កំពុងធ្វេការងារ","គាត់កំពុងធ្វើការងារ"),

#  # Third batch
#  ("ខ្ញុំចូលចិតសិក្សាភាសា","ខ្ញុំចូលចិត្តសិក្សាភាសា"),
#  ("ខ្ញុំទៅសាលារៀនរាល់ថ្ងែ","ខ្ញុំទៅសាលារៀនរាល់ថ្ងៃ"),
#  ("គាត់ជាមិត្តល្អរបសខ្ញុំ","គាត់ជាមិត្តល្អរបស់ខ្ញុំ"),
#  ("ខ្ញុំកំពុងរៀនសម្រាប់ប្រឡ","ខ្ញុំកំពុងរៀនសម្រាប់ប្រឡង"),
#  ("សិស្សចូលចិតសិក្សា","សិស្សចូលចិត្តសិក្សា"),
#  ("ថ្ងៃនេះយើងរៀនមេរៀនថ្ម","ថ្ងៃនេះយើងរៀនមេរៀនថ្មី"),
#  ("ខ្ញុំចូលចិតអានរឿង","ខ្ញុំចូលចិត្តអានរឿង"),
#  ("ពួកគេកំពុងរៀនជាមយគ្រូ","ពួកគេកំពុងរៀនជាមួយគ្រូ"),
#  ("សាលារៀននេះល្បីណ","សាលារៀននេះល្បីណាស់"),
#  ("គាត់ចូលចិតសិក្សាខ្មែរ","គាត់ចូលចិត្តសិក្សាខ្មែរ"),

#  # Fourth batch
#  ("ខ្ញុំសរសេរអត្ថបទមិនត្រិម","ខ្ញុំសរសេរអត្ថបទមិនត្រឹម"),
#  ("សិស្សរៀនមេរៀនថ្ម","សិស្សរៀនមេរៀនថ្មី"),
#  ("ពួកគេរៀននសាលា","ពួកគេរៀននៅសាលា"),
#  ("គាត់ចូលចិតសៀវភៅថ្ម","គាត់ចូលចិត្តសៀវភៅថ្មី"),
#  ("ខ្ញុំកំពុងធ្វេលំហាត់","ខ្ញុំកំពុងធ្វើលំហាត់"),
#  ("សិស្សចូលចិតរៀន","សិស្សចូលចិត្តរៀន"),
#  ("គាត់ចូលចិតភាសាខ្មែរ","គាត់ចូលចិត្តភាសាខ្មែរ"),
#  ("ខ្ញុំរៀននសាកលវិទ្យាល័យ","ខ្ញុំរៀននៅសាកលវិទ្យាល័យ"),
#  ("សាលារៀននៅក្បែផ្ទះ","សាលារៀននៅក្បែរផ្ទះ"),
#  ("ពួកគេចូលចិតលេងកីឡា","ពួកគេចូលចិត្តលេងកីឡា"),

#  # Fifth batch
#  ("ខ្ញុំចូលចិតសរសេរ","ខ្ញុំចូលចិត្តសរសេរ"),
#  ("សិស្សរៀនភាសាអង់គ្លេសស","សិស្សរៀនភាសាអង់គ្លេស"),
#  ("គាត់កំពុងធ្វេការស្រាវជ្រាវ","គាត់កំពុងធ្វើការស្រាវជ្រាវ"),
#  ("ខ្ញុំចូលចិតការអាន","ខ្ញុំចូលចិត្តការអាន"),
#  ("សិស្សកំពុងរៀនមេរៀ","សិស្សកំពុងរៀនមេរៀន"),
#  ("ពួកគេចូលចិតការសិក្សា","ពួកគេចូលចិត្តការសិក្សា"),
#  ("ខ្ញុំកំពុងរៀននផ្ទះ","ខ្ញុំកំពុងរៀននៅផ្ទះ"),
#  ("សិស្សចូលចិតសៀវភៅ","សិស្សចូលចិត្តសៀវភៅ"),
#  ("គាត់ចូលចិតអាន","គាត់ចូលចិត្តអាន"),
#  ("ពួកគេកំពុងធ្វេការសិក្សា","ពួកគេកំពុងធ្វើការសិក្សា")
# ]

# # Evaluation
# correct_count = 0
# total = len(evaluation_pairs)

# print("Starting evaluation of", total, "test cases...")
# print("-" * 50)

# for i, (wrong, correct) in enumerate(evaluation_pairs, 1):
#  # Extract the sentence part (remove "fix spelling: " prefix if present)
#  if wrong.startswith("fix spelling: "):
#     input_sentence = wrong.replace("fix spelling: ", "")
#  else:
#     input_sentence = wrong

#  # Get prediction
#  pred = correct_khmer(input_sentence)

#  # Compare (remove trailing punctuation if needed for comparison)
#  pred_clean = pred.rstrip(" ៕។៖,;")
#  correct_clean = correct.rstrip(" ៕។៖,;")

#  if pred_clean == correct_clean:
#     correct_count += 1
#     status = "✓"
#  else:
#     status = "✗"

#  # Print detailed results
#  print(f"{i}. {status}")
#  print(f" Input: {input_sentence}")
#  print(f" Expected: {correct}")
#  print(f" Got: {pred}")
#  print()

# # Calculate and print accuracy
# accuracy = (correct_count / total) * 100
# print("=" * 50)
# print(f"Accuracy: {correct_count}/{total} = {accuracy:.2f}%")
# print("=" * 50)

# # Optional: Print summary of errors
# if correct_count < total:
#  print("\nErrors encountered:")
#  print("-" * 30)
#  for i, (wrong, correct) in enumerate(evaluation_pairs):
#     if wrong.startswith("fix spelling: "):
#         input_sentence = wrong.replace("fix spelling: ", "")
#     else:
#         input_sentence = wrong
#     pred = correct_khmer(input_sentence)
#     pred_clean = pred.rstrip(" ៕។៖,;")
#     correct_clean = correct.rstrip(" ៕។៖,;")

#     if pred_clean != correct_clean:
#         print(f"\n{i+1}. Input: {input_sentence}")
#         print(f" Expected: {correct}")
#         print(f" Got: {pred}")